In [4]:
import pandas as pd, numpy as np, glob

path = glob.glob("/kaggle/input/**/GC_in_5_minute_new.csv", recursive=True)[0]
df = pd.read_csv(path, parse_dates=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)
df["date"] = df["datetime"].dt.date
print(path, df.shape, df["datetime"].min(), df["datetime"].max())

/kaggle/input/datasets/youneseloiarm/comex-gold-futures-dataset-gc-contract/GC_in_5_minute_new.csv (6097, 8) 2025-09-14 22:00:00 2025-10-15 00:00:00


In [5]:
i_optF, i_fLvg, i_maxLoss, atrLen, atrMult, cap0 = 0.21, 1.0, 9794.92, 14, 2.0, 100000.0
leverage, tick_size, slip_ticks, comm_pct = 1.0, 0.10, 2, 0.02 
slip = slip_ticks * tick_size

df["ema9"] = df["close"].ewm(span=9, adjust=False).mean()
vwap_num = (df["close"]*df["volume"]).groupby(df["date"]).cumsum()
vwap_den = df["volume"].groupby(df["date"]).cumsum()
df["vwap"] = vwap_num/vwap_den
tr = pd.concat([df["high"]-df["low"], (df["high"]-df["close"].shift()).abs(), (df["low"]-df["close"].shift()).abs()], axis=1).max(axis=1)
df["atr"] = tr.ewm(alpha=1/atrLen, adjust=False).mean()
df["longX"]  = (df["ema9"]>df["vwap"]) & (df["ema9"].shift()<=df["vwap"].shift())
df["shortX"] = (df["ema9"]<df["vwap"]) & (df["ema9"].shift()>=df["vwap"].shift())

def qty_now(eq):
    base = (eq*i_optF*i_fLvg/i_maxLoss) if i_maxLoss>0 else 0.0
    return max(base, 0.0) * leverage 

def open_pos(side, price, qty, equity):
    fill = price + slip if side=="L" else price - slip
    equity -= fill*qty*comm_pct/100
    return {"side":side,"entry":fill,"qty":qty,"peak":fill,"stop":None}, equity

def close_pos(pos, price, equity, trades):
    fill = price - slip if pos["side"]=="L" else price + slip
    pnl = (fill-pos["entry"])*pos["qty"] if pos["side"]=="L" else (pos["entry"]-fill)*pos["qty"]
    comm = fill*pos["qty"]*comm_pct/100
    equity += pnl - comm
    trades.append(pnl-comm)
    return equity

equity, trades, pos, eq_curve = cap0, [], None, []

for i in range(len(df)):
    r = df.iloc[i]
    if pos:
        trailD = r["atr"]*atrMult
        if pos["side"]=="L":
            pos["peak"] = max(pos["peak"], r["high"])
            if pos["peak"]-pos["entry"] >= trailD: pos["stop"] = pos["peak"]-trailD
            if pos["stop"] is not None and r["low"]<=pos["stop"]:
                equity = close_pos(pos, pos["stop"], equity, trades); pos=None
        else:
            pos["peak"] = min(pos["peak"], r["low"])
            if pos["entry"]-pos["peak"] >= trailD: pos["stop"] = pos["peak"]+trailD
            if pos["stop"] is not None and r["high"]>=pos["stop"]:
                equity = close_pos(pos, pos["stop"], equity, trades); pos=None

    if pos and ((pos["side"]=="L" and r["shortX"]) or (pos["side"]=="S" and r["longX"])):
        equity = close_pos(pos, r["close"], equity, trades); pos=None

    if pos is None:
        q = qty_now(equity)
        if q>0 and r["longX"]:
            pos, equity = open_pos("L", r["close"], q, equity)
        elif q>0 and r["shortX"]:
            pos, equity = open_pos("S", r["close"], q, equity)
    eq_curve.append(equity)

df["equity"] = eq_curve
trades = pd.Series(trades)
print(f"trades={len(trades)}  final_equity={equity:,.2f}")

trades=137  final_equity=99,701.85


In [6]:
eq = df.set_index("datetime")["equity"]
daily = eq.resample("D").last().dropna()
dret = daily.pct_change().dropna()

ret_total = eq.iloc[-1]/cap0 - 1
years = (eq.index[-1]-eq.index[0]).days/365.25
cagr = (eq.iloc[-1]/cap0)**(1/years)-1 if years>0 else np.nan
maxdd = ((eq - eq.cummax())/eq.cummax()).min()
sharpe = dret.mean()/dret.std()*np.sqrt(252) if dret.std()>0 else np.nan
gp, gl = trades[trades>0].sum(), -trades[trades<0].sum()
pf = gp/gl if gl>0 else np.nan
expectancy = trades.mean()

win_max = lose_max = w = l = 0
for t in trades:
    if t>0: w+=1; l=0
    else: l+=1; w=0
    win_max, lose_max = max(win_max,w), max(lose_max,l)

monthly = daily.resample("ME").last().pct_change().dropna()

print(f"Return:        {ret_total:.2%}")
print(f"CAGR:          {cagr:.2%}")
print(f"Sharpe:        {sharpe:.2f}")
print(f"Max Drawdown:  {maxdd:.2%}")
print(f"Profit Factor: {pf:.2f}")
print(f"Expectancy:    {expectancy:.2f}")
print(f"Max Win Streak:  {win_max}")
print(f"Max Lose Streak: {lose_max}")
print("\nMonthly Returns:")
print((monthly*100).round(2).to_string())

Return:        -0.30%
CAGR:          -3.57%
Sharpe:        -5.03
Max Drawdown:  -0.39%
Profit Factor: 0.91
Expectancy:    -0.54
Max Win Streak:  5
Max Lose Streak: 7

Monthly Returns:
datetime
2025-10-31    0.05
Freq: ME
